In [20]:
import pandas as pd
import glob

baemin_files = sorted(glob.glob("../data/baemin/baemin*.csv"))
baemin_files

coupang_files = sorted(glob.glob("../data/coupang/coupang*.csv"))
coupang_files

['../data/coupang\\coupang20261001.csv',
 '../data/coupang\\coupang20261002.csv',
 '../data/coupang\\coupang20261003.csv',
 '../data/coupang\\coupang20261004.csv',
 '../data/coupang\\coupang20261005.csv',
 '../data/coupang\\coupang20261006.csv',
 '../data/coupang\\coupang20261007.csv',
 '../data/coupang\\coupang20261008.csv',
 '../data/coupang\\coupang20261009.csv',
 '../data/coupang\\coupang20261010.csv',
 '../data/coupang\\coupang20261011.csv',
 '../data/coupang\\coupang20261012.csv',
 '../data/coupang\\coupang20261013.csv',
 '../data/coupang\\coupang20261014.csv',
 '../data/coupang\\coupang20261015.csv',
 '../data/coupang\\coupang20261016.csv',
 '../data/coupang\\coupang20261017.csv',
 '../data/coupang\\coupang20261018.csv',
 '../data/coupang\\coupang20261019.csv',
 '../data/coupang\\coupang20261020.csv',
 '../data/coupang\\coupang20261021.csv',
 '../data/coupang\\coupang20261022.csv',
 '../data/coupang\\coupang20261023.csv',
 '../data/coupang\\coupang20261024.csv',
 '../data/coupan

In [21]:
# 파일 합치기
baemin_df = []

for bm in baemin_files:
    b_df = pd.read_csv(bm)
    # 칼럼 재지정
    b_df.columns = ["주문일시", "전체금액", "광고유입여부","광고집행비(원)"]
    b_df["플랫폼"] = "배민"
    baemin_df.append(b_df)

baemin = pd.concat(baemin_df, ignore_index = True) 

coupang_df = []

for cp in coupang_files:
    c_df = pd.read_csv(cp)
    c_df = c_df.rename(columns ={
        "order_time" : "주문일시", 
        "total_price" : "전체금액",
        "advertisement_yn" : "광고유입여부",
        "ad_pct" : "광고집행비(원)",

        })

    c_df["광고집행비(원)"]  = c_df["전체금액"] * c_df["광고집행비(원)"] / 100
    c_df["플랫폼"] = "쿠팡"   
    c_df = c_df[["주문일시", "전체금액", "광고유입여부", "광고집행비(원)", "플랫폼"]]
    coupang_df.append(c_df)

coupang = pd.concat(coupang_df, ignore_index= True)

#  데이터 통합

delivery = pd.concat([baemin, coupang],ignore_index = True )

# 날짜

delivery["주문일시"] = pd.to_datetime(delivery["주문일시"])
delivery["주문일자"] = delivery["주문일시"].dt.date 
delivery = delivery.sort_values("주문일시",ignore_index = True)
# ignore index로 인덱스 정리
delivery.head()

,주문일시,전체금액,광고유입여부,광고집행비(원),플랫폼,주문일자
0,2026-10-01 11:46:07,18000,Y,1800.0,쿠팡,2026-10-01
1,2026-10-01 11:48:34,26000,Y,700.0,배민,2026-10-01
2,2026-10-01 11:53:22,15000,N,0.0,배민,2026-10-01
3,2026-10-01 12:06:50,21000,Y,2100.0,쿠팡,2026-10-01
4,2026-10-01 12:16:29,58000,Y,5800.0,쿠팡,2026-10-01


In [22]:
delivery_pivot = delivery.pivot_table(
    index = "주문일자",
    columns = "플랫폼",
    values = "전체금액",
    aggfunc = "sum"
    
)
delivery_pivot["합계"] = delivery_pivot.sum(axis=1)

delivery_pivot

플랫폼,배민,쿠팡,합계
주문일자,,,
2026-10-01,959000,562000,1521000
2026-10-02,1348000,904000,2252000
2026-10-03,1294000,779000,2073000
2026-10-04,1127000,928000,2055000
2026-10-05,1103000,543000,1646000
2026-10-06,1028000,576000,1604000
2026-10-07,1227000,549000,1776000
2026-10-08,946000,439000,1385000
2026-10-09,1200000,835000,2035000


In [23]:
# 광고유입 비율
ad_order = (
delivery[delivery["광고유입여부"] == "Y"].groupby("플랫폼").size()

)

total_order = delivery.groupby("플랫폼").size()
ad_ratio = ad_order / total_order *100

# 플랫폼별 집행비
ad_cost = delivery.groupby("플랫폼")["광고집행비(원)"].sum()
delivery

print("광고 유입 비율: ",ad_ratio)
print("플랫폼별 집행비: ",ad_cost)

광고 유입 비율:  플랫폼
배민    34.978355
쿠팡    51.966292
dtype: float64
플랫폼별 집행비:  플랫폼
배민    243170.0
쿠팡    886700.0
Name: 광고집행비(원), dtype: float64


In [24]:
bank = pd.read_csv("../data/bank_20261001_20261031.csv", encoding = "cp949")
bank_df = bank[bank["거래처"] == "식자재마트"]
bank_df["식자재비"] = bank_df.pop("출금액")

bank_df["거래일시"] = pd.to_datetime(bank_df["거래일시"]).dt.date
food_cost=bank_df.groupby("거래일시")["식자재비"].sum()
food_cost 
# food_cost.rename_axis("주문일자")
# 여기서 미리 바꾸기도 가능



거래일시
2026-10-01    197100
2026-10-02    380400
2026-10-03    247600
2026-10-04    289300
2026-10-05    214400
2026-10-06    209500
2026-10-07    332800
2026-10-08    219300
2026-10-09    351300
2026-10-10    324900
2026-10-11    225000
2026-10-12    305500
2026-10-13    314100
2026-10-14    375200
2026-10-15    288500
2026-10-16    389700
2026-10-17    342000
2026-10-18    184600
2026-10-19    250100
2026-10-20    205600
2026-10-21    292400
2026-10-22    263500
2026-10-23    289100
2026-10-24    289800
2026-10-25    312300
2026-10-26    272800
2026-10-27    200400
2026-10-28    391000
2026-10-29    213300
2026-10-30    302800
2026-10-31    397300
Name: 식자재비, dtype: int64

In [25]:
result = pd.concat([delivery_pivot,food_cost],axis = 1)
result = result.reset_index()
result = result.rename(columns={"index" : "주문일자"})
result["매출-식자재비"] = result["합계"] - result["식자재비"]
result

,주문일자,배민,쿠팡,합계,식자재비,매출-식자재비
0,2026-10-01,959000,562000,1521000,197100,1323900
1,2026-10-02,1348000,904000,2252000,380400,1871600
2,2026-10-03,1294000,779000,2073000,247600,1825400
3,2026-10-04,1127000,928000,2055000,289300,1765700
4,2026-10-05,1103000,543000,1646000,214400,1431600
5,2026-10-06,1028000,576000,1604000,209500,1394500
6,2026-10-07,1227000,549000,1776000,332800,1443200
7,2026-10-08,946000,439000,1385000,219300,1165700
8,2026-10-09,1200000,835000,2035000,351300,1683700
9,2026-10-10,1379000,719000,2098000,324900,1773100


In [26]:
result.to_csv("result.csv",index=False,encoding="utf-8-sig")
